# End 2 End notebook for fingerprint classification project
This notebook eseentially does : 
1. Explore dataset contains from existing directories
2. Creates Train / Validation / Test datasets 
3. Create classification model with a pre-trained MobileNet / EfficientNet version with the following architecture
   MobileNet / EfficientNet -> Hidden Dense Layers -> Classification Layer
4. Carries out Transfer Learning with frozen pre-trained based model
5. Carries out Fine-Tuning with unfrozen layers of the pre-trained base model 
6. Evaluation and comparison of both transfer learning and fine tuning models

## Project Pre-requisites

In [ ]:
# Import libraries
import os
from pathlib import Path
import random
import itertools

import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks

from tensorflow.keras.utils import image_dataset_from_directory

from sklearn.metrics import confusion_matrix, classification_report

## Project Hyper-parameters
Principal knobs to parametrize this projet 

1. Datasets parameters
   1. Number of classes i.e. fingers to consider for the classification problem
   2. split ratio for validation and test datasets, per default set to 50/50 for validation vs. test
2. Model Architecture parameters
   1. MobileNet / Efficient version
   2. Number of neurons in first hidden dense layer, the second dense layer uses this number divived by 4
      (other architectures with a single dense layer and 3 distinct dense layers have been explored but number of dense layers is not parametrizable)
   3. activation function of hidden dense layers is set to leak_relu with slope = 0.001
      (other activation functions relu, gelu, gelu_approx, leaky_rely with slope = 0.2 have been explore)
   4. Dropout rate of layers associated with hidden dense layers in set to 0.3 
      (no exaploration has been carried out with lower values, to avoid overfitting)
3. Training paramters
   1. Number of epochs, patience for both transfer learning and fine tuning

In [ ]:
# pre-trained model selection
# "MobileNet", "MobileNetV2", "MobileNetV3Large", "EfficientNetB0, "EfficientNetV2B0", "EfficientNetB3"
PreTrained_Model_Name = "MobileNetV2"

if PreTrained_Model_Name == "MobileNet":
    from tensorflow.keras.applications import MobileNet
    from tensorflow.keras.applications.mobilenet import preprocess_input
elif PreTrained_Model_Name == "MobileNetV2":
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
elif PreTrained_Model_Name == "MobileNetV3Large":
    from tensorflow.keras.applications import MobileNetV3Large
    from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
elif PreTrained_Model_Name == "EfficientNetB0":
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.applications.efficientnet import preprocess_input
elif PreTrained_Model_Name == "EfficientNetV2B0":
    from tensorflow.keras.applications import EfficientNetV2B0
    from tensorflow.keras.applications.efficientnet import preprocess_input
elif PreTrained_Model_Name == "EfficientNetB3":
    from tensorflow.keras.applications import EfficientNetB3
    from tensorflow.keras.applications.efficientnet import preprocess_input
else:
    print("Unknown pre-trained model name")
    exit(0)

In [ ]:
# Check selected pre-trained model
print(f"Selected pre-trained model version : {PreTrained_Model_Name}")

In [ ]:
# Configuration GPU
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU(s) détecté(s): {len(gpus)} - Croissance mémoire activée")
    else:
        print("⚠️  Aucun GPU détecté - Utilisation du CPU (entraînement sera lent)")
except Exception as e:
    print(f"Configuration GPU: {e}")

print(f"\n📦 Versions des bibliothèques :")
print(f"  - TensorFlow : {tf.__version__}")
print(f"  - Keras      : {keras.__version__}")
print(f"  - NumPy      : {np.__version__}")

print(f"\n🚀 Prêt pour le transfer learning en computer vision !")

## Project Constants

In [ ]:
# Random seeding 
TENSORFLOW_RANDOM_SEED = 42
tf.random.set_seed(TENSORFLOW_RANDOM_SEED)

NUMPY_RANDOM_SEED = 42
np.random.seed(NUMPY_RANDOM_SEED)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Dataset selection
DATASET_ROOTDIR_PATH = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Custom" 
print(f"📁 Fingerprints dataset images root directory path : {DATASET_ROOTDIR_PATH}")

# labels
# class_names = ['thumb', 'index', 'middle', 'ring', 'little']
class_names = ['thumb', 'index', 'middle', 'ring', 'little']

num_classes = len(class_names)
print(f"\n👁️ {num_classes} fingerprints classes : {class_names}")

# model name
fingerprint_model_base_name = PreTrained_Model_Name + "_FingerPrint_Model"
fingerprint_model_name = fingerprint_model_base_name + "_" + str(num_classes) + "_classes"
print(f"Fingerprint model name : {fingerprint_model_name}")

## 1. Dataset Exploration
The original dataset SOCOFing was found from kaggle @ https://www.kaggle.com/datasets/mreccentric/socofing-full-modified/data
It contains roughly 55k, greyscaled and (96, 103) images, of which 
- 6k original images
- 49k low-medium-high altered original images (cut, blur, drop), examples below

A subset of the above is been considered for training / validation / test as follow : 
- 6k original images, 1.2k images by class
- 18k low altered images : 6k zcut, 6k blur and 6k drop, 3.6k images by class
- Total of 24k images, 4.8 images by class

Considering that altered images are actually augmented for the training steps, this subset has then been manually split to
mimic the train / validation / test separation as follow:

For each finger class (1.2K images), randomly split original images into two sets, i.e. train = 800 and validation/test = 400
For each of the images in the train set, collect the corresponding altered images, i.e. cut, blur and drop. 
Then split validation and test using 50/50 ratio in this notebook 

This creates the following dataset for each class : 
- train = 800 + 3 * 800 = 3200 images
- validation = 200 images
- test = 200 images

To sum up, the dataset contains 18k images, with 3.6k images by class 

In [ ]:
# Distribution graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]] # For displaying purposes, pick columns that have between 1 and 50 unique values
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int((nCol + nGraphPerRow - 1) / nGraphPerRow)
    plt.figure(num = None, figsize = (6 * nGraphPerRow, 8 * nGraphRow), dpi = 80, facecolor = 'w', edgecolor = 'k')
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        columnDf = df.iloc[:, i]
        if (not np.issubdtype(type(columnDf.iloc[0]), np.number)):
            valueCounts = columnDf.value_counts()
            valueCounts.plot.bar()
        else:
            columnDf.hist()
        plt.ylabel('counts')
        plt.xticks(rotation = 90)
        plt.title(f'{columnNames[i]} (column {i})')
    plt.tight_layout(pad = 1.0, w_pad = 1.0, h_pad = 1.0)
    plt.show()

In [ ]:
# utility function to collect images and classes
def traverse_dataset_dir(dataset_dir, class_names):
    '''
        return a dataframe containing image files information and corresponding classes from the dataset directory
            dataset_dir : directory to scan
            class_names : labels to consider
    '''
    # List to store file information
    imgdata = []

    # Traverse the input directory to get all image files
    for dirname, _, filenames in os.walk(dataset_dir):

        # check whether image is in a class which ought to be considered
        try:
            basename = os.path.basename(dirname)
            val = class_names.index(basename)
        except ValueError:
            # class not found
            continue

        # Traverse the image files
        for filename in filenames:
         
            if filename == ".DS_Store":  # Skip macOS system files
                continue

            # identifies class 
            if filename.find("thumb") != -1:
                finger = "thumb"
            elif filename.find("index") != -1:
                finger = "index"
            elif filename.find("middle") != -1:
                finger = "middle"
            elif filename.find("ring") != -1:
                finger = "ring"
            elif filename.find("little") != -1:
                finger = "little"
            else:
                finger = None

            # identifies alteration type from filename
            if filename.find("Zcut") != -1:
                feature = 'zcut'
            elif filename.find('CR') != -1:
                feature = 'cr'
            elif filename.find('Obl') != -1:
                feature = 'obl'
            else:
                feature = 'unaltered'

            # image is to be considered given the current class names 
            file_path = os.path.join(dirname, filename)
    
            try:
                img = Image.open(file_path)
                img.verify()  # Verify that the file is a valid image
            except:
                print(f"Invalid image file: {file_path}")
                continue

            imgdata.append([file_path, img.format, img.size, img.mode, feature, finger])

    # Create a DataFrame
    imgdf = pd.DataFrame(imgdata, columns=["ImagePath", "ImageFormat", "ImageSize", "ImageMode", "Feature", "Finger"])

    # Add an index column
    imgdf.index.name = "ID"

    return imgdf

In [ ]:
# explore train dataset
dataset_train_dir = DATASET_ROOTDIR_PATH + "/train"
imgdf = traverse_dataset_dir(dataset_train_dir, class_names)

In [ ]:
# display the dataframe info
imgdf.info()

In [ ]:
# check null values
imgdf.isnull().sum()

In [ ]:
# Display the distribution of Feature, Finger
print(imgdf["Feature"].value_counts())
print(imgdf["Finger"].value_counts())

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
print(f"Percentage of Thumbs: {imgdf['Finger'].value_counts().get('thumb', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Index fingers: {imgdf['Finger'].value_counts().get('index', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Middle fingers: {imgdf['Finger'].value_counts().get('middle', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Ring fingers: {imgdf['Finger'].value_counts().get('ring', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Little fingers: {imgdf['Finger'].value_counts().get('little', 0) / len(imgdf) * 100:.2f}%")

In [ ]:
# Display of number of features
print(f"Percentage of unaltered: {imgdf['Feature'].value_counts().get('unaltered', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Zcut: {imgdf['Feature'].value_counts().get('zcut', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of CR: {imgdf['Feature'].value_counts().get('cr', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Obl: {imgdf['Feature'].value_counts().get('obl', 0) / len(imgdf) * 100:.2f}%")

In [ ]:
plotPerColumnDistribution(imgdf, 10, 5)

In [ ]:
# visualisation of some images
dataset_target_dir = DATASET_ROOTDIR_PATH + "/train/"
print(f"FingerPrint Examples from dataset dir {dataset_target_dir}\n")

num_examples_images = 10
fig, axes = plt.subplots(num_classes, num_examples_images, figsize=(18, 3 * num_classes))

for i, class_name in enumerate(class_names):
    class_path = Path(os.path.join(dataset_target_dir, class_name))
    image_files = list(class_path.glob('*.*'))[:num_examples_images]
    
    for j, img_path in enumerate(image_files):
        img = plt.imread(img_path)
        axes[i, j].imshow(img)
        axes[i, j].axis('on')
        if j == 0:
            axes[i, j].set_xlabel(class_name.upper(), fontsize=12, fontweight='bold', rotation=0, ha='right', va='center')

plt.suptitle("Fingerprint image example by class", fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
# this requires a later version of numpy which clashes wtih tensorflow
import cv2

# utility function to apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to enhance the contrast of fingerprint images
def preprocess_CLAHE(image):
    '''
        Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to enhance the contrast of the input image.
        This can help improve the visibility of fingerprint features, especially in low-contrast images.
        
        Parameters:
            image (tf.Tensor): Input image tensor with pixel values in the range [0, 255].
        
        Returns:
            tf.Tensor: Image tensor after applying CLAHE, with pixel values in the range [0, 255].
    '''
    # Convert the image to uint8 format for OpenCV processing
    image_uint8 = image.astype(np.uint8)
    
    # Convert the image from RGB to L channel (grayscale) for CLAHE processing
    gray_image = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2GRAY)

    resized_gray_image = cv2.resize(gray_image, (128, 128), interpolation=cv2.INTER_LANCZOS4)  # Resize to match model input size

    # Apply CLAHE to the L channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
    clahe_image = clahe.apply(resized_gray_image)

    # Convert the image from L channel back to RGB color space
    rgb_image_clahe = cv2.cvtColor(clahe_image, cv2.COLOR_GRAY2RGB)
    
    # Convert the processed image back to float32 format for TensorFlow processing
    return rgb_image_clahe.astype(np.float32)

In [ ]:
# visualisation of some images after applying CLAHE
dataset_target_dir = DATASET_ROOTDIR_PATH + "/train/"
print(f"FingerPrint Examples from dataset dir {dataset_target_dir} with CLAHE preprocessing\n")

num_examples_images = 10
fig, axes = plt.subplots(num_classes, num_examples_images, figsize=(18, 3 * num_classes))

for i, class_name in enumerate(class_names):
    class_path = Path(os.path.join(dataset_target_dir, class_name))
    image_files = list(class_path.glob('*.*'))[:num_examples_images]
    
    for j, img_path in enumerate(image_files):
        img = plt.imread(img_path)
        img_clahe = preprocess_CLAHE(img)
        img_clahe = img_clahe.astype(np.uint8)
        axes[i, j].imshow(img_clahe)
        axes[i, j].axis('on')
        if j == 0:
            axes[i, j].set_xlabel(class_name.upper(), fontsize=12, fontweight='bold', rotation=0, ha='right', va='center')

plt.suptitle("Fingerprint image example by class", fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
# validation/test dataset
dataset_val_dir = DATASET_ROOTDIR_PATH + "/validation"
imgdf = traverse_dataset_dir(dataset_val_dir, class_names)

In [ ]:
# display the dataframe info
imgdf.info()

In [ ]:
# check null values
imgdf.isnull().sum()

In [ ]:
# Display the distribution of Feature, Finger
print(imgdf["Feature"].value_counts())
print(imgdf["Finger"].value_counts())

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
print(f"Percentage of Thumbs: {imgdf['Finger'].value_counts().get('thumb', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Index fingers: {imgdf['Finger'].value_counts().get('index', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Middle fingers: {imgdf['Finger'].value_counts().get('middle', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Ring fingers: {imgdf['Finger'].value_counts().get('ring', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Little fingers: {imgdf['Finger'].value_counts().get('little', 0) / len(imgdf) * 100:.2f}%")

In [ ]:
# Display of feature
print(f"Percentage of unaltered: {imgdf['Feature'].value_counts().get('unaltered', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Zcut: {imgdf['Feature'].value_counts().get('zcut', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of CR: {imgdf['Feature'].value_counts().get('cr', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Obl: {imgdf['Feature'].value_counts().get('obl', 0) / len(imgdf) * 100:.2f}%")

In [ ]:
plotPerColumnDistribution(imgdf, 10, 5)

In [ ]:
# visualisation of some images
dataset_target_dir = DATASET_ROOTDIR_PATH + "/validation/"
print(f"FingerPrint Examples from dataset dir {dataset_target_dir}\n")

num_examples_images = 10
fig, axes = plt.subplots(num_classes, num_examples_images, figsize=(18, 3 * num_classes))

for i, class_name in enumerate(class_names):
    class_path = Path(os.path.join(dataset_target_dir, class_name))
    image_files = list(class_path.glob('*.*'))[:num_examples_images]
    
    for j, img_path in enumerate(image_files):
        img = plt.imread(img_path)
        axes[i, j].imshow(img)
        axes[i, j].axis('on')
        if j == 0:
            axes[i, j].set_xlabel(class_name.upper(), fontsize=12, fontweight='bold', rotation=0, ha='right', va='center')

plt.suptitle("Fingerprint image example by class", fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation of some images
dataset_target_dir = DATASET_ROOTDIR_PATH + "/validation/"
print(f"FingerPrint Examples from dataset dir {dataset_target_dir} with CLAHE preprocessing\n")

num_examples_images = 10
fig, axes = plt.subplots(num_classes, num_examples_images, figsize=(18, 3 * num_classes))

for i, class_name in enumerate(class_names):
    class_path = Path(os.path.join(dataset_target_dir, class_name))
    image_files = list(class_path.glob('*.*'))[:num_examples_images]
    
    for j, img_path in enumerate(image_files):
        img = plt.imread(img_path)
        img_clahe = preprocess_CLAHE(img)
        img_clahe = img_clahe.astype(np.uint8)
        axes[i, j].imshow(img_clahe)
        axes[i, j].axis('on')
        if j == 0:
            axes[i, j].set_xlabel(class_name.upper(), fontsize=12, fontweight='bold', rotation=0, ha='right', va='center')

plt.suptitle("Fingerprint image example by class", fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

## 2. Create datasets for training / validation / test

In [ ]:
print("🔧 Datasets creation for training and validation/test ...\n")

# Dataset creation parameters
BATCH_SIZE = 64
IMAGE_SIZE = (96, 96)

# Create traning dataset with class filtering
dataset_train_dir = DATASET_ROOTDIR_PATH + "/train"
train_ds = image_dataset_from_directory(
    dataset_train_dir,
    image_size=IMAGE_SIZE,
    batch_size=None,  # Load all images in one batch for proper preprocessing, batching will be done later after preprocessing
    label_mode='categorical',  # One-hot encoding for multi-class
    class_names=class_names  # Explicitly specify the class names to ensure correct mapping
)

# Create validation dataset with the same class names to ensure correct mapping
dataset_val_dir = DATASET_ROOTDIR_PATH + "/validation"
val_test_ds = image_dataset_from_directory(
    dataset_val_dir,
    image_size=IMAGE_SIZE,
    batch_size=None,  # Load all images in one batch for proper preprocessing, batching will be done later after preprocessing
    label_mode='categorical',
    class_names=class_names  # Same order of classes
)

# split validation dataset into 2 using 50/50 ratio for validation and test datasets
val_batches = tf.data.experimental.cardinality(val_test_ds)
test_ds_cardinality = int(val_batches * 50 / 100)

test_ds = val_test_ds.take(test_ds_cardinality)
test_ds.class_names = class_names
val_ds = val_test_ds.skip(test_ds_cardinality)
val_ds.class_names = class_names

print(f"✅ Datasets created !\n")

print(f"📊 Training, validation and datasets information:")
print(f"  - Training Dataset Classes        : {train_ds.class_names}")
print(f"  - Validation Dataset Classes      : {val_ds.class_names}")
print(f"  - Test Dataset Classes            : {test_ds.class_names}")

# Checking number of classes in both datasets
if len(train_ds.class_names) != num_classes:
    print(f"\n⚠️  Error : {len(train_ds.class_names)} classes found instead of {num_classes} in training dataset !")
    print(f"   Classes found: {train_ds.class_names}\n")
elif len(val_test_ds.class_names) != num_classes:
    print(f"\n⚠️  Error : {len(val_test_ds.class_names)} classes found instead of {num_classes} in validation dataset !")
    print(f"   Classes found : {val_test_ds.class_names}")
else:
    print(f"  - ✅ Number of correct classes : {num_classes} \n")

print(f"  - Training (1 batch)    : {len(train_ds)}")
print(f"  - Validation (1 batch)  : {len(val_ds)}")
print(f"  - Test (1 batch)        : {len(test_ds)}")

# 3. Train / validation / test datasets preparation for training
    This consists in normalisation and shuffling (training only)
    (no augmentation of train dataset is carried out since altered images are already embeded in it)

In [ ]:
print("🎨 Fingerprint dataset preparation : normalisation and shuffling (training only) ...\n")

def preprocess_pipeline(image):
    '''
        Preprocessing pipeline to apply CLAHE and normalisation to the input image.
        This function will be applied to each image in the dataset.
        
        Parameters:
            image (tf.Tensor): Input image tensor with pixel values in the range [0, 255].
            label (tf.Tensor): Input label tensor.
        
        Returns:
            tuple: Preprocessed image tensor and label tensor.
    '''
    # Apply CLAHE to enhance the contrast of fingerprint images
    image_clahe = tf.numpy_function(preprocess_CLAHE, [image], tf.float32)
    
    # Critical: set the shape of the output tensor to match the input image shape
    image_clahe.set_shape((128, 128, 3))  # Set the shape to (height, width, channels) after CLAHE processing
    
    # Normalisation is required for MobileNet / EfficientNet pre-trained models
    # This is carried out with keras.utils.preprocess_input
    image_normalized = preprocess_input(image_clahe)
    
    return image_normalized

# Dataset preparation function
def prepare_dataset(ds, shuffle=True):

    # preprocess images with CLAHE and normalisation 
    normalization = lambda x, y: (preprocess_pipeline(x), y)
    ds = ds.map(normalization, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Shuffle (training only)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    
    return ds

# Apply transformations
train_ds_prepared = prepare_dataset(train_ds, shuffle=True)
val_ds_prepared = prepare_dataset(val_ds, shuffle=False)
test_ds_prepared = prepare_dataset(test_ds, shuffle=False)

print("✅ Dataset preparation completed !\n")

print("📝 Applied transformations :")
print("  - Training dataset   : CLAHE + Normalisation + shuffle")
print("  - Validation dataset : CLAHE + Normalisation")
print("  - Test dataset       : CLAHE + Normalisation")

print("\n  - Caching et prefetch activated for performance")

In [ ]:
# Now create the batches for training, validation and test datasets
# add caching and prefetching for performance optimization
train_ds_prepared = train_ds_prepared.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds_prepared = val_ds_prepared.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds_prepared = test_ds_prepared.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"📊 Training, validation and test datasets information after preparation:")
print(f"  - Training (1 batch)    : {len(train_ds_prepared)}")
print(f"  - Validation (1 batch)  : {len(val_ds_prepared)}")
print(f"  - Test (1 batch)        : {len(test_ds_prepared)}")
print(f"  - Batch size            : {BATCH_SIZE}")
print(f"  - Training samples      : ~{len(train_ds_prepared) * BATCH_SIZE}")
print(f"  - Validation samples    : ~{len(val_ds_prepared) * BATCH_SIZE}")
print(f"  - Test samples          : ~{len(test_ds_prepared) * BATCH_SIZE}")

## 4. Build FingerPrint Model
Pre-trained MobileNet/EfficientNet-based model + Hidden dense layers + Classifier layer

In [ ]:
# Constants for the model
INPUT_SHAPE = (128, 128, 3)
NUM_CLASSES = num_classes

# Size of intermediate dense layers between base-model and classifier stage
HIDDEN_DENSE_LAYERS_SIZE = [256, 64]

# activation function
def leaky_relu(x):
    return keras.activations.relu(x, negative_slope=0.01)

ACTIVATION_FUNCTION = leaky_relu

# Regularization setting
L2_REGULARIZATION = 0.01
DROPOUT_RATE = 0.3

# Learning Rate Standard for Transfer Learning of frozen model 
LEARNING_RATE = 0.001

In [ ]:
print(f"📥 Loading {PreTrained_Model_Name} ...")

PreTrained_FingerPrint_Model_basename = PreTrained_Model_Name + "_FingerPrint_BaseModel"

# Loading pre-trained MobileNet without the classifier (include_top=False)
if PreTrained_Model_Name == "MobileNet":
    base_model = MobileNet(
        name=PreTrained_FingerPrint_Model_basename,
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
elif PreTrained_Model_Name == "MobileNetV2":
    base_model = MobileNetV2(
        name=PreTrained_FingerPrint_Model_basename,
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
elif PreTrained_Model_Name == "MobileNetV3Large":
    base_model = MobileNetV3Large(
        name=PreTrained_FingerPrint_Model_basename,
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
elif PreTrained_Model_Name == "EfficientNetB0":
    base_model = EfficientNetB0(
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
elif PreTrained_Model_Name == "EfficientNetV2B0":
    base_model = EfficientNetV2B0(
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
elif PreTrained_Model_Name == "EfficientNetB3":
    base_model = EfficientNetB3(
        input_shape=INPUT_SHAPE,
        include_top=False,  # Exclure la couche de classification ImageNet
        weights='imagenet',  # Load pre-trained weights from ImageNet
        pooling='avg'  # Global Average Pooling to end model
    )
else:
    print(f"Unknown pre-trained model version : {PreTrained_Model_Name}")
    exit(0)

print(f"✅ {PreTrained_Model_Name} model loaded !\n")

# Display model structure
print(f"📋 Model Structure - {PreTrained_Model_Name} :")
print(f"  - Architecture   : {PreTrained_Model_Name}")
print(f"  - Input shape    : {INPUT_SHAPE}")
print(f"  - Weights        : ImageNet (1000 classes)")
print(f"  - Output shape   : {base_model.output_shape}")
print(f"  - Number of layers : {len(base_model.layers)}")

# Count parameters
total_params = base_model.count_params()
print(f"\n  - Total number of parameters : {total_params:,}")
print(f"  - Memory size (approx.) : {total_params * 4 / (1024**2):.1f} MB")

In [ ]:
print("🧊 Construct FingerPrint model with hidden dense layers and classifier...\n")

# Freeze the base model to keep the pre-trained features
base_model.trainable = False

print(f"✅ Base model frozen")
print(f"   {len(base_model.layers)} layers not trainable\n")

# Input layer and pre-trained model
inputs = keras.Input(shape=INPUT_SHAPE)
x = base_model(inputs, training=False)

# Smoothened out hidden layers with dropout (avoid overfitting) and regularizerization 
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(HIDDEN_DENSE_LAYERS_SIZE[0], 
                 activation=ACTIVATION_FUNCTION, 
                 kernel_regularizer=keras.regularizers.l2(L2_REGULARIZATION))(x)

x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(HIDDEN_DENSE_LAYERS_SIZE[1], 
                 activation=ACTIVATION_FUNCTION, 
                 kernel_regularizer=keras.regularizers.l2(L2_REGULARIZATION))(x)

# Check number of classes
if num_classes > 2:
    classification_activation = 'softmax'
else:
    classification_activation = 'sigmoid'

# Smootened out classification layer
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(NUM_CLASSES, activation=classification_activation)(x)

fingerprint_model = keras.Model(inputs, outputs, name=fingerprint_model_name)

# Compile the model with Adam optimizer and standard LR  
fingerprint_model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', 
             keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

print("✅ Model compiled !\n")

print(f"📊 Architecture {PreTrained_Model_Name} based FingerPrint:")
print(f"  Input{INPUT_SHAPE} → {PreTrained_Model_Name} → Dropout({DROPOUT_RATE}) → Dense({HIDDEN_DENSE_LAYERS_SIZE[0]},{ACTIVATION_FUNCTION}) → Dropout({DROPOUT_RATE}) → Dense({HIDDEN_DENSE_LAYERS_SIZE[1]},{ACTIVATION_FUNCTION}) → Dropout({DROPOUT_RATE}) → Dense({NUM_CLASSES},{classification_activation})\n")

trainable = sum([tf.size(w).numpy() for w in fingerprint_model.trainable_weights])
non_trainable = sum([tf.size(w).numpy() for w in fingerprint_model.non_trainable_weights])

print(f"  - Trainable parameters     : {trainable:,}")
print(f"  - Non-trainable parameters : {non_trainable:,}")
print(f"  - Total                    : {trainable + non_trainable:,}")

print("\n💡 FringerPrint Model configuration:")
print(f"  - Learning rate           : {LEARNING_RATE} (standard)")
print(f"  - Dropout rate            : {DROPOUT_RATE} (medium)")
print(f"  - 1rst Hidden dense layer : {HIDDEN_DENSE_LAYERS_SIZE[0]} neurons")
print(f"  - 2nd Hidden dense layer  : {HIDDEN_DENSE_LAYERS_SIZE[1]} neurons")
print(f"  - L2 regularization       : {L2_REGULARIZATION}")

## Display FingerPrint Model Architecture

In [ ]:
# utility to display layer output shape
def get_output_shape(layer):
    try:
        if hasattr(layer, 'output_shape'):
            return str(layer.output_shape)
        elif hasattr(layer, 'output'):
            return str(layer.output.shape)
        else:
            return "N/A"
    except:
        return "N/A"

# Display the first 10 and last 10 layers of the base model to understand the feature extraction process
def display_model_layers(model, num_input_layers, num_output_layers):

    print(f"🔍 Display {model.name} model architecture\n")
    print("="*100)

    print("\n📌 First layers :")
    for i, layer in enumerate(model.layers[:num_input_layers]):
        output_shape = get_output_shape(layer)
        print(f"  {i+1:3d}. {layer.name:40s} | Output: {output_shape:30s} | Trainable: {layer.trainable}")

    print("\n      ...")

    print("\n📌 Last layers :")
    for i, layer in enumerate(model.layers[-num_output_layers:], start=len(base_model.layers)-num_output_layers):
        output_shape = get_output_shape(layer)
        print(f"  {i+1:3d}. {layer.name:40s} | Output: {output_shape:30s} | Trainable: {layer.trainable}")

    print("\n" + "="*100)

In [ ]:
# display base model layers 
display_model_layers(base_model, 10, 10)

In [ ]:
display_model_layers(fingerprint_model, 10, 10)

In [ ]:
print(f"🔍 Visualizing {fingerprint_model.name} FingerPrint model architecture")
print("="*100)

fingerprint_model.summary()

print("="*100)

##  5. Frozen Model Transfer Learning Training

In [ ]:
# Constants for training callbacks
EARLYSTOPPING_PATIENCE = 15
EARLYSTOPPING_VERBOSE = 1

REDUCELRONPLATEAU_FACTOR = 0.5
REDUCELRONPLATEAU_PATIENCE = 4
REDUCELRONPLATEAU_MIN_LR = 1e-7
REDUCELRONPLATEAU_VERBOSE = 1

fingerprint_model_dir = "/Users/laurent/Projects/projet-fingerprint-validator/models/"
MODELCHECKPOINT_PATH =  fingerprint_model_dir + fingerprint_model_name + "_Transfer_Learning.keras"
MODELCHECKPOINT_VERBOSE = 1

FIT_NUM_EPOCHS = 90
FIT_VERBOSE = 1

In [ ]:
# Transfer Learning Training function
def transfer_learning_fingerprint_model(model: keras.Model, train_ds: tf.data.Dataset, val_ds: tf.data.Dataset):

    print("🚀 Training (frozen) FingerPrint model ...\n")

    # Define callbacks
    early_stopping = callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=EARLYSTOPPING_PATIENCE,
        restore_best_weights=True,
        mode='max',
        verbose=EARLYSTOPPING_VERBOSE
    )

    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=REDUCELRONPLATEAU_FACTOR,
        patience=REDUCELRONPLATEAU_PATIENCE,
        min_lr=REDUCELRONPLATEAU_MIN_LR,
        verbose=REDUCELRONPLATEAU_VERBOSE
    )

    checkpoint = callbacks.ModelCheckpoint(
        MODELCHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=MODELCHECKPOINT_VERBOSE
    )

    print(f"Configuration :")
    print(f"  - Maximum Number of Epochs : {FIT_NUM_EPOCHS}")
    print(f"  - EarlyStopping Patience   : {EARLYSTOPPING_PATIENCE} epochs")
    print(f"  - Reduce LR Patience       : {REDUCELRONPLATEAU_PATIENCE} epochs")
    print(f"  - LR initial               : {LEARNING_RATE}")
    print(f"  - LR min                  : {REDUCELRONPLATEAU_MIN_LR}\n")
    
    history_fit = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FIT_NUM_EPOCHS,
        callbacks=[early_stopping, reduce_lr, checkpoint],
        verbose=FIT_VERBOSE
    )
    
    print("\n✅ Transfer Learning (frozen) model training completed !")
    print(f"   Epochs completed : {len(history_fit.history['loss'])}")
    
    best_frozen_val_acc = max(history_fit.history['val_accuracy'])
    best_frozen_epoch = history_fit.history['val_accuracy'].index(best_frozen_val_acc) + 1
    print(f"   Best validation accuracy : {best_frozen_val_acc*100:.2f}% (epoch {best_frozen_epoch})")

    if best_frozen_val_acc < 0.70:
        print(f"\n⚠️  Warning : Accuracy of {best_frozen_val_acc*100:.1f}% is below 70%")
    elif best_frozen_val_acc < 0.80:
        print(f"\n✅ Good : Accuracy of {best_frozen_val_acc*100:.1f}%")
    else:
        print(f"\n🎯 Excellent : Accuracy of {best_frozen_val_acc*100:.1f}%")

    return history_fit, best_frozen_val_acc

In [ ]:
# transfer learning training with frozen base model
history_frozen, best_frozen_val_acc = transfer_learning_fingerprint_model(fingerprint_model, 
                                                                          train_ds_prepared, 
                                                                          val_ds_prepared)

## Transfer Learning Frozen Model Evaluation

In [ ]:
def make_prediction_samples(model, test_ds, num_predictions=20):

    print(f"🔍 Predictions samples of {model} model\n")
    print("="*100)

    for images, labels in test_ds.take(1):
        # make prediction
        predictions = model.predict(images[:num_predictions], verbose=0)
    
    print("\n📊 Prediction :")
    print("-"*100)
    
    for i in range(num_predictions):

        example_predictions = np.copy(predictions[i])

        true_label = np.argmax(labels[i])
        top1_pred_label = np.argmax(example_predictions)
        top1_confidence = example_predictions[top1_pred_label]

        example_predictions[top1_pred_label] = 0
        top2_pred_label = np.argmax(example_predictions)
        top2_confidence = example_predictions[top2_pred_label]

        status = "✅" if true_label == top1_pred_label else "❌"
        print(f"{status} Image {i+1}: Label={class_names[true_label]:12s} | Prediction={class_names[top1_pred_label]:12s} | Confidence={top1_confidence*100:5.1f}% | 2nd Prediction={class_names[top2_pred_label]:12s} | 2nd Confidence={top2_confidence*100:5.1f}%")

    print("\n" + "="*100)

In [ ]:
# Make a few predictions using test dataset
make_prediction_samples(fingerprint_model, test_ds_prepared)

In [ ]:
# Evaluation using test dataset
print("📊 Evaluation of transfer learning frozen model using test dataset...\n")

results_test_model_frozen = fingerprint_model.evaluate(test_ds_prepared, verbose=0)
transfer_learning_test_accuracy = results_test_model_frozen[1]
transfer_learning_test_top2_accuracy = results_test_model_frozen[2]

print("Results :")
print(f"  - Loss                : {results_test_model_frozen[0]:.4f}")
print(f"  - Accuracy            : {transfer_learning_test_accuracy*100:.2f}%")
print(f"  - Top-2 Accuracy      : {transfer_learning_test_top2_accuracy*100:.2f}%")

# check overfitting
# Evaluation using validation data
results_val_model_frozen = fingerprint_model.evaluate(val_ds_prepared, verbose=0)
transfer_learning_val_accuracy = results_val_model_frozen[1]

print(f"\nCheck overfitting :")
print(f"  - Validation Accuracy : {transfer_learning_val_accuracy*100:.2f}%")
print(f"  - Test Accuracy       : {transfer_learning_test_accuracy*100:.2f}%")

In [ ]:
def display_training_history(training_history, title):

    # Display history training graphs
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Loss
    axes[0].plot(training_history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(training_history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(training_history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[1].plot(training_history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(alpha=0.3)

    # Top2 Accuracy
    axes[2].plot(training_history.history['top2_accuracy'], label='Train Top-2Accuracy', linewidth=2)
    axes[2].plot(training_history.history['val_top2_accuracy'], label='Val Top-2Accuracy', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Top2 Accuracy', fontsize=12)
    axes[2].set_title('Top2 Accuracy', fontsize=14, fontweight='bold')
    axes[2].legend(fontsize=11)
    axes[2].grid(alpha=0.3)

    plt.suptitle(f'{fingerprint_model.name} model training {title}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# display transfer learning history 
display_training_history(history_frozen, 'with features extraction (Frozen)')

## 6. Fine Tuning Training

In [ ]:
# Constants
frozen_fingerprint_basemodel = base_model
frozen_fingerprint_model = fingerprint_model

# Number of layers in the model 
NUM_BASE_MODEL_LAYERS = len(frozen_fingerprint_basemodel.layers)

# keep ratio of 40/60 unfrozen layers vs. frozen layers
UNFROZEN_FROZEN_LAYERS_RATIO = 0.33
NUM_MODEL_UNFROZEN_BASE_MODEL_LAYERS = int(NUM_BASE_MODEL_LAYERS * UNFROZEN_FROZEN_LAYERS_RATIO)

#  ratio of unfrozen first layers vs. unfrozen last layers
UNFROZEN_FROZEN_INPUT_OUTPUT_LAYERS_RATIO = 0
FINE_TUNING_UNFROZEN_INPUT_BASE_MODEL_LAYERS = int(NUM_MODEL_UNFROZEN_BASE_MODEL_LAYERS * UNFROZEN_FROZEN_INPUT_OUTPUT_LAYERS_RATIO)
FINE_TUNING_UNFREEZE_OUTPUT_BASE_MODEL_LAYERS = int(NUM_MODEL_UNFROZEN_BASE_MODEL_LAYERS - FINE_TUNING_UNFROZEN_INPUT_BASE_MODEL_LAYERS)

# Low LR for fine tuning
FINE_TUNE_LR = 1e-5 

In [ ]:
print("🔥 Fine-Tuning Preparation...\n")

# APPROCHE LA PLUS SIMPLE : Modifier le modèle frozen en place
print(f"💡 Strategy : unfreeze the {PreTrained_Model_Name} pre-trained base model and continue training\n")
print(f"Number of layers in the base model                      : {NUM_BASE_MODEL_LAYERS}")
print(f"Number of the first unfrozen layers in the base model   : {FINE_TUNING_UNFROZEN_INPUT_BASE_MODEL_LAYERS}")
print(f"Number of the last unfrozen layers in the base model   : {FINE_TUNING_UNFREEZE_OUTPUT_BASE_MODEL_LAYERS}")

# Unfreeze base model
frozen_fingerprint_basemodel.trainable = True

# Unfreeze the first FINE_TUNING_UNFREEZE_INPUT_BASE_MODEL_LAYERS-th layers
# Unfreeze the last FINE_TUNING_UNFREEZE_OUTPUT_BASE_MODEL_LAYERS-th layers

for i, layer in enumerate(frozen_fingerprint_basemodel.layers):
    if i < FINE_TUNING_UNFROZEN_INPUT_BASE_MODEL_LAYERS:
        layer.trainable = True
    elif i < (NUM_BASE_MODEL_LAYERS - FINE_TUNING_UNFREEZE_OUTPUT_BASE_MODEL_LAYERS):
        layer.trainable = False
    else:
        layer.trainable = True


print(f"📊 Unfreeze configuration :")
print(f"  - Frozen layers   : {NUM_MODEL_UNFROZEN_BASE_MODEL_LAYERS}")
print(f"  - Unfrozen layers : {len(frozen_fingerprint_basemodel.layers) - NUM_MODEL_UNFROZEN_BASE_MODEL_LAYERS}")

# Model params counting 
trainable_count = sum([tf.size(w).numpy() for w in frozen_fingerprint_basemodel.trainable_weights])
non_trainable_count = sum([tf.size(w).numpy() for w in frozen_fingerprint_basemodel.non_trainable_weights])
total = trainable_count + non_trainable_count

print(f"  - Trainable params : {trainable_count:,} ({trainable_count/total*100:.1f}%)")
print(f"  - Frozen params    : {non_trainable_count:,}\n")

# Recompile model 
frozen_fingerprint_model.compile(
    optimizer=optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', 
             keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

# Fine-tuned model with partial frozen base model 
finetuned_fingerprint_model = frozen_fingerprint_model

print(f"✅ Fingerprint model {finetuned_fingerprint_model.name} recompiled with low LR = {FINE_TUNE_LR}")

In [ ]:
# display base model unfrozen layers 
display_model_layers(frozen_fingerprint_basemodel, 10, FINE_TUNING_UNFREEZE_OUTPUT_BASE_MODEL_LAYERS + 2)

In [ ]:
# display model params 
finetuned_fingerprint_model.summary()

### Fine Tuning Training

In [ ]:
# Constants for training callbacks
EARLYSTOPPING_PATIENCE = 30
EARLYSTOPPING_VERBOSE = 1

REDUCELRONPLATEAU_FACTOR = 0.5
REDUCELRONPLATEAU_PATIENCE = 7
REDUCELRONPLATEAU_MIN_LR = 1e-8
REDUCELRONPLATEAU_VERBOSE = 1

fingerprint_model_name = PreTrained_Model_Name + "_FingerPrint_Model_" + str(num_classes) + "_classes_Fine_Tuning.keras"
MODELCHECKPOINT_PATH = fingerprint_model_dir + "/" + fingerprint_model_name
MODELCHECKPOINT_VERBOSE = 1

FIT_NUM_EPOCHS = 120
FIT_VERBOSE = 1

In [ ]:
# Fine Tuning Training function
def fine_tuning_fingerprint_model(model: keras.Model, train_ds: tf.data.Dataset, val_ds: tf.data.Dataset):

    print("🚀 Fine-tuning FringerPrint Model...\n")

    # Callbacks 
    early_stopping_ft = callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=EARLYSTOPPING_PATIENCE,
        restore_best_weights=True,
        mode='max',
        verbose=EARLYSTOPPING_VERBOSE
    )

    reduce_lr_ft = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=REDUCELRONPLATEAU_FACTOR,
        patience=REDUCELRONPLATEAU_PATIENCE,
        min_lr=REDUCELRONPLATEAU_MIN_LR,
        verbose=REDUCELRONPLATEAU_VERBOSE
    )

    checkpoint_ft = callbacks.ModelCheckpoint(
        MODELCHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=MODELCHECKPOINT_VERBOSE
    )

    print(f"Training for {FIT_NUM_EPOCHS} epochs maximum...\n")

    history_finetuned = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FIT_NUM_EPOCHS,
        callbacks=[early_stopping_ft, reduce_lr_ft, checkpoint_ft],
        verbose=FIT_VERBOSE
    )

    print("\n✅ Fine-tuning complete !")
    print(f"   Number of epochs completed : {len(history_finetuned.history['loss'])}")

    # Best performances
    best_finetuned_val_acc = max(history_finetuned.history['val_accuracy'])
    best_finetuned_epoch = history_finetuned.history['val_accuracy'].index(best_finetuned_val_acc) + 1
    print(f"   Best validation accuracy : {best_finetuned_val_acc*100:.2f}% (epoch {best_finetuned_epoch})")

    if best_finetuned_val_acc < 0.70:
        print(f"\n⚠️  Warning : Accuracy of {best_finetuned_val_acc*100:.1f}% is below 70%")
    elif best_finetuned_val_acc < 0.80:
        print(f"\n✅ Good : Accuracy of {best_finetuned_val_acc*100:.1f}%")
    else:
        print(f"\n🎯 Excellent : Accuracy of {best_finetuned_val_acc*100:.1f}%")

    return history_finetuned, best_finetuned_val_acc

In [ ]:
# fine tuning training
history_finetuned, best_finetuned_val_acc = fine_tuning_fingerprint_model(finetuned_fingerprint_model, 
                                                                          train_ds_prepared, 
                                                                          val_ds_prepared)

In [ ]:
# Basic Comparaison
improvement = (best_finetuned_val_acc - best_frozen_val_acc) * 100

print("\n📊 Comparison Transfer Learning vs. Fine Tuning (best validation accuracy only):")
print(f" - Transfer Learning    : {best_frozen_val_acc*100:.1f}%")
print(f" - Fine-tuned           : {best_finetuned_val_acc*100:.1f}%")
print(f" - Improvement          : {improvement:+.1f}%")

In [ ]:
# Make a few predictions using test dataset
make_prediction_samples(finetuned_fingerprint_model, test_ds_prepared)

In [ ]:
# Evaluation using test dataset
print("📊 Evaluation of fine tuned model using test dataset...\n")

results_test_model_fine_tuning= finetuned_fingerprint_model.evaluate(test_ds_prepared, verbose=0)
fine_tuning_test_accuracy = results_test_model_fine_tuning[1]
fine_tuning_test_top2_accuracy = results_test_model_fine_tuning[2]

print("Results :")
print(f"  - Loss                : {results_test_model_fine_tuning[0]:.4f}")
print(f"  - Accuracy            : {fine_tuning_test_accuracy*100:.2f}%")
print(f"  - Top-2 Accuracy      : {fine_tuning_test_top2_accuracy*100:.2f}%")

# check overfitting
# Evaluation using validation data
results_val_fine_tuning = finetuned_fingerprint_model.evaluate(val_ds_prepared, verbose=0)
fine_tuning_val_accuracy = results_val_fine_tuning[1]

print(f"\nCheck overfitting :")
print(f"  - Validation Accuracy : {fine_tuning_val_accuracy*100:.2f}%")
print(f"  - Test Accuracy       : {fine_tuning_test_accuracy*100:.2f}%")

In [ ]:
# display fine tuning history 
display_training_history(history_finetuned, 'with fine tuning (Unfrozen)')

## 7. Compare Transfer Learning and Fine Tuning Training Results

In [ ]:
# Graph for comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracies comparison
metrics = ['Accuracy', 'Top-2 Accuracy']
frozen_scores = [results_test_model_frozen[1]*100, results_test_model_frozen[2]*100]
finetuned_scores = [results_test_model_fine_tuning[1]*100, results_test_model_fine_tuning[2]*100]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, frozen_scores, width, label='Feature Extraction', color='skyblue', edgecolor='black', linewidth=2)
axes[0].bar(x + width/2, finetuned_scores, width, label='Fine-Tuning', color='lightcoral', edgecolor='black', linewidth=2)
axes[0].set_ylabel('Score (%)', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3, axis='y')
axes[0].set_ylim([0, 100])

# Ajouter les valeurs sur les barres
for i, (f, ft) in enumerate(zip(frozen_scores, finetuned_scores)):
    axes[0].text(i - width/2, f + 1, f'{f:.1f}%', ha='center', fontsize=10, fontweight='bold')
    axes[0].text(i + width/2, ft + 1, f'{ft:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Comparaison des loss
categories = ['Validation Loss']
frozen_loss = [results_test_model_frozen[0]]
finetuned_loss = [results_test_model_fine_tuning[0]]

x2 = np.arange(len(categories))
axes[1].bar(x2 - width/2, frozen_loss, width, label='Feature Extraction', color='skyblue', edgecolor='black', linewidth=2)
axes[1].bar(x2 + width/2, finetuned_loss, width, label='Fine-Tuning', color='lightcoral', edgecolor='black', linewidth=2)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Loss Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(categories)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3, axis='y')

# Ajouter les valeurs
axes[1].text(x2[0] - width/2, frozen_loss[0] + 0.02, f'{frozen_loss[0]:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].text(x2[0] + width/2, finetuned_loss[0] + 0.02, f'{finetuned_loss[0]:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Transfer Learning vs Fine-Tuning : Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
print("📊 Confusion Matrix and Detailed Analysis...\n")

# Get the predictions
y_true = []
y_pred = []

for images, labels in test_ds_prepared:
    predictions = finetuned_fingerprint_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Visualize
plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Fine-Tuned Model', fontsize=16, fontweight='bold', pad=20)
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha='right', fontsize=11)
plt.yticks(tick_marks, class_names, fontsize=11)

# Add values in cells
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], 'd'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black",
             fontsize=12, fontweight='bold')

plt.ylabel('Label', fontsize=13, fontweight='bold')
plt.xlabel('Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Dump classification report
print("\n📄 Detailed classification report :\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

### Detailed Analysis by Class

In [ ]:
# Compute accuracy by class
class_accuracies = cm.diagonal() / cm.sum(axis=1)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if acc > 0.85 else '#f39c12' if acc > 0.75 else '#e74c3c' for acc in class_accuracies]
bars = ax.bar(class_names, class_accuracies * 100, color=colors, edgecolor='black', linewidth=2)

# Add bar values
for bar, acc in zip(bars, class_accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{acc*100:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Classes', fontsize=13, fontweight='bold')
ax.set_title('Accuracy by Classe - Fine-Tuned Model', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim([0, 105])
ax.grid(alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Identify best and worst classes
best_class_idx = np.argmax(class_accuracies)
worst_class_idx = np.argmin(class_accuracies)

print(f"\n🏆 Best class : {class_names[best_class_idx]} ({class_accuracies[best_class_idx]*100:.1f}%)")
print(f"⚠️  Worst class : {class_names[worst_class_idx]} ({class_accuracies[worst_class_idx]*100:.1f}%)")

## 8. Prepare FingerPrint model for production

In [ ]:
# prediction function
def predict_fingerprint(model, image_input_size, fingerprint_image_path, class_names, topk_predictions):
    """
    Predict the fingerprint class given an image with fine)tuned model
    
    Args:
        model: fingerprint trained model
        image_input_size: size of images expected for the trained model
        fingerprint_image_path: fingerprint image path
        class_names: classes names
        topk_predictions: integer for reporting top-k predictions
    """

    try:
        # Load and prep image for prediction
        img = tf.keras.preprocessing.image.load_img(fingerprint_image_path, target_size=image_input_size)

        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)  # Add batch dimension
        img_array = preprocess_input(img_array)  # Normalisation with MobileNet / EfficientNet preprocessing
    except Exception as e:
        print(f"Error loading or processing image : {e}")
        return(0)
    
    # Prediction
    predictions = model.predict(img_array, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx]
    
    # Report top-k prédictions
    topk_idx = np.argsort(predictions[0])[-topk_predictions:][::-1]
    
    return {
        'predicted_class': class_names[predicted_class_idx],
        'confidence': confidence,
        'all_probabilities': predictions[0],        
        'top-k': [(class_names[i], predictions[0][i]) for i in topk_idx]
    }

print("✅ Prediction function created !")

In [ ]:
print("🔮 Predictions using randomly selected images...\n")

num_examples_prediction = 10

# Select randomly a few images
test_images_predictions = []

for class_name in class_names:
    class_path = Path(os.path.join(DATASET_ROOTDIR_PATH + "/validation", class_name))
    images = list(class_path.glob('*.*'))

    # Randomly select num_examples images in each class
    test_images_predictions.extend(random.sample(images, min(num_examples_prediction, len(images))))

fig, axes = plt.subplots(len(class_names), num_examples_prediction, figsize=(20, 10))
axes = axes.flatten()

for idx, img_path in enumerate(test_images_predictions[:num_examples_prediction * len(class_names)]):
    # Predict
    result = predict_fingerprint(finetuned_fingerprint_model, IMAGE_SIZE, img_path, class_names, 2)
    
    # Display image
    img = plt.imread(img_path)
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    # Vraie classe (depuis le nom du dossier)
    filename = os.path.basename(img_path)
    if filename.find("thumb") != -1:
        true_class = "thumb"
    elif filename.find("index") != -1:
        true_class = "index"
    elif filename.find("middle") != -1:
        true_class = "middle"
    elif filename.find("ring") != -1:
        true_class = "ring"
    elif filename.find("little") != -1:
        true_class = "little"
    else:
        true_class = None
    
    predicted_class = result['predicted_class']
    confidence = result['confidence']
    
    # Color-code result : green-correct vs. red-incorrect 
    color = 'green' if true_class == predicted_class else 'red'
    
    # Title
    title = f"Class: {true_class}\nPrediction: {predicted_class}\nConfidence: {confidence*100:.1f}%"
    axes[idx].set_title(title, fontsize=10, color=color, fontweight='bold')

plt.suptitle('Prediction using randomly selected images', fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

### VIsualize all probabilities for a few selected images

In [ ]:
# Select few images from the above set
num_examples_probabilities = 10

# Randomly select num_examples images in each class
test_images_probabilities = random.sample(test_images_predictions, num_examples_probabilities)

fig, axes = plt.subplots(num_examples_probabilities, 2, figsize=(16, 16))

for idx, img_path in enumerate(test_images_probabilities[:num_examples_probabilities]):

    sample_image_path = test_images_probabilities[idx]
    result = predict_fingerprint(finetuned_fingerprint_model, IMAGE_SIZE, sample_image_path, class_names, 2)

    # Image
    img = plt.imread(sample_image_path)
    axes[idx, 0].imshow(img)
    axes[idx, 0].axis('off')
    axes[idx, 0].set_title(f"Image : {sample_image_path.parent.name}", fontsize=14, fontweight='bold')

    # Probabilities
    probs = result['all_probabilities'] * 100
    colors = ['#2ecc71' if i == np.argmax(probs) else '#3498db' for i in range(len(probs))]

    bars = axes[idx, 1].barh(class_names, probs, color=colors, edgecolor='black', linewidth=2)
    axes[idx, 1].set_xlabel('Probability (%)', fontsize=13, fontweight='bold')
    axes[idx, 1].set_title('Probability Distribution', fontsize=14, fontweight='bold')
    axes[idx, 1].grid(alpha=0.3, axis='x')

    # Add values
    for bar, prob in zip(bars, probs):
        width = bar.get_width()
        axes[idx, 1].text(width + 1, bar.get_y() + bar.get_height()/2.,
                    f'{prob:.1f}%',
                    ha='left', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()